# Journal de bord

## 📅 Journal de bord – Jour 1 (04/08/2026)

### 🔧 Étapes / Actions réalisées
- Lecture de tous les documents du cas d'usage : mail de cadrage, sujet d'examen (Bank Marketing), canevas `canvas-cas-usage-v2.ipynb`, journal de bord, feuille de route, référentiel de compétences C1-C9.
- Mise en place de l'espace de travail : copie du canevas et du journal en fichiers de travail, copie du dataset.
- Début de la Phase 1 (cadrage, §1.1-§1.2) : identification du client (banque de détail) et de son problème métier (coût et mauvais ciblage du démarchage téléphonique).
- Réflexion sur le moment où la prédiction doit intervenir dans le processus métier (avant l'appel). Ce point conditionnera le traitement de la variable `duration`.

### 📝 Observations / Difficultés
- Pas de blocage technique, le travail en est encore au cadrage.
- Reste à trancher la formulation exacte de l'énoncé du problème IA (§1.2).

### ✅ Prochaines étapes
- Finaliser §1 (bénéficiaires, critères de succès, risques éthiques anticipés).
- Démarrer §2 (identification et dictionnaire des données).

## 📅 Journal de bord – Jour 2 (15/08/2026)

### 🔧 Étapes / Actions réalisées
- Énoncé du problème IA finalisé (§1.2).
- Fin de la Phase 1 (§1) : bénéficiaires directs/indirects, critères de succès (métier/modèle/opérationnel), risques éthiques anticipés. Distinction entre les variables sensibles pour un usage commercial (âge, job, situation familiale, éducation) et les catégories particulières au sens de l'article 9 du RGPD (aucune variable du dataset n'en fait partie).
- Phase 2 (§2) : description des sources de données et dictionnaire complet des 21 variables (type, signification, valeurs possibles).
- Repérage des variables sensibles et de `duration` comme fuite d'information potentielle : la durée de l'appel n'est connue qu'après l'appel, alors que la prédiction doit être faite avant.

### 📝 Observations / Difficultés
- La distinction « sensible pour l'usage » / « catégorie particulière RGPD » m'a pris du temps, mais sans elle l'analyse éthique qui suit aurait été fragile.
- Comme le moment de la prédiction avait déjà été posé le 04/08, la fuite sur `duration` a été facile à repérer.

### ✅ Prochaines étapes
- Structurer le dépôt et l'environnement avant l'EDA.

## 📅 Journal de bord – Jour 3 (23/08/2026)

### 🔧 Étapes / Actions réalisées
- Création du dépôt Git (repo privé), avec la même arborescence que pour les modules précédents (`notebooks/`, `src/`, `models/`), environnement virtuel avec `uv` + `requirements.txt`.
- Correction du §0.5 (traçabilité), resté à l'état de template depuis la première séance.
- Phase 3 (§3, EDA), qualité des données : 12 doublons, modalités `unknown` jusqu'à 20.9 % sur `default`.
- `pdays` quasi dégénérée : 96.3 % des lignes à la valeur sentinelle 999.
- Forte colinéarité entre les indicateurs socio-économiques (`emp.var.rate`, `euribor3m`, `nr.employed`, corrélations > 0.9).
- Déséquilibre de la cible confirmé (11.3 % / 88.7 %).
- Relation en U entre l'âge et le taux de souscription, redondance entre `job` et `education`.

### 📝 Observations / Difficultés
- Pas de blocage. Reprendre l'arborescence des modules précédents m'a évité de repenser l'organisation.
- Les résultats de l'EDA (redondance `job`/`education`, poids de l'âge) donnent des éléments concrets pour justifier le Scénario 3 (retrait des variables sensibles).

### ✅ Prochaines étapes
- Phase 4 (§4) : split stratifié, pipeline de préparation sans fuite, définition des 4 scénarios de variables.

## 📅 Journal de bord – Jour 4 (29/08/2026)

### 🔧 Étapes / Actions réalisées
- Split train/test stratifié fait avant tout traitement (§4).
- Pipeline de préparation (encodage, mise à l'échelle) ajusté uniquement sur le train.
- Définition des 4 scénarios de variables (avec/sans `duration`, avec/sans variables sensibles).
- Changement de traitement pour `pdays` : la colonne avait d'abord été supprimée à cause de la valeur sentinelle 999, puis recodée en valeur manquante + imputation par la médiane (calculée sur le train). La suppression faisait perdre l'information de récence pour les ~3.7 % de clients déjà recontactés.
- Analyse de `job`/`education` et calcul du Disparate Impact (règle des 4/5, vue en M2-B1) pour chiffrer le risque de discrimination : DI job = 0.219, DI âge = 0.182.
- Nuance retenue : un écart dans les données historiques (comportement des clients) n'est pas en soi une discrimination de la part de la banque. Le risque se situe dans l'usage d'un modèle automatisé pour choisir qui appeler (boucle de rétroaction possible).
- Graphiques de l'EDA refaits (titres, axes, tri, palette) pour les rendre lisibles.

### 📝 Observations / Difficultés
- La première solution pour `pdays` tenait la route mais perdait de l'information utile.
- Les deux DI sont très en dessous du seuil de 0.8 : l'argument pour le Scénario 3 est maintenant chiffré.
- Les premiers graphiques étaient lisibles pour moi mais pas pour un lecteur extérieur.

### ✅ Prochaines étapes
- Phase 5 (§5) : benchmark multi-modèles en validation croisée.

## 📅 Journal de bord – Jour 5 (05/09/2026)

### 🔧 Étapes / Actions réalisées
- Méthode retenue pour `duration` : comparer les scénarios avec et sans `duration`, et expliquer l'écart par le moment où la donnée est connue.
- Phase 5 (§5) : benchmark régression logistique + Random Forest sur les 4 scénarios, en validation croisée stratifiée 5-fold.
- Petite exploration des hyperparamètres du Random Forest (3 configurations, même CV que le benchmark).
- Évaluation finale sur le test set.
- Phase 6 (§6) : tableau comparatif, mesure de la latence d'inférence (200 prédictions chronométrées), recommandation finale.

### 📝 Observations / Difficultés
- Résultat principal : sans `duration`, le F1 baisse de 28 % (0.642 → 0.462), ce qui confirme la fuite. Retirer en plus les variables sensibles ne coûte presque rien (0.462 → 0.464). C'est ce chiffre qui a tranché.
- Bug trouvé en Phase 6 : les mêmes objets modèles (`LogisticRegression`/`RandomForestClassifier`) étaient réutilisés d'un scénario à l'autre, donc chaque entraînement écrasait celui du scénario précédent. Les métriques affichées restaient justes (prédiction faite juste après l'entraînement), mais pas le pipeline stocké pour la suite. Corrigé avec `sklearn.base.clone()`.

### ✅ Prochaines étapes
- Analyse éthique et réglementaire (RGPD, biais/équité sur les prédictions du modèle retenu et pas seulement sur les données brutes).
- Vérifier ensuite la cohérence de la démarche avec les modules déjà suivis.

## 📅 Journal de bord – Jour 6 (13/09/2026)

### 🔧 Étapes / Actions réalisées
- Analyse éthique complétée au niveau M7-B1 : tableau de Disparate Impact sur plusieurs variables (âge, job, situation familiale, éducation), avec comparaison entre étiquettes réelles et prédictions pour voir si le modèle amplifie les écarts.
- Intersectionnalité `age × job` : DI = 0.050, écart ×20 entre groupes.
- Taux d'erreur (FNR/FPR) et calibration par sous-groupe, pour identifier qui est lésé.
- Partie RGPD : nature des données, différence pseudonymisation/anonymisation, base légale envisagée, analyse de l'article 22.
- Qualification AI Act (risque minimal) avec la grille officielle à 4 niveaux.
- Couverture des 4 catégories de risques de la compétence C2 (biais, inclusion, sécurité, environnement). Pour l'environnement, l'argument est le choix du modèle le plus sobre.
- Tableau récapitulatif des risques classés par priorité, et liste de questions à poser au DPO.
- Datasheet du dataset au format Gebru et al. (7 sections), en complément du notebook (`datasheet.md`).

### 📝 Observations / Difficultés
- Le DI calculé sur les prédictions montre une amplification du biais sur 3 des 4 variables sensibles (toutes sauf `education`), alors qu'elles ont été retirées des entrées du modèle.
- Pour l'article 22, il a fallu bien séparer aide à la décision et décision entièrement automatisée.
- Une partie de la datasheet reprend des éléments du §2 ; l'intérêt est d'avoir un format standard réutilisable.

### ✅ Prochaines étapes
- Vérifier la cohérence de la démarche avec les ressources des modules déjà suivis et combler les manques.

## 📅 Journal de bord – Jour 7 (19/09/2026)

### 🔧 Étapes / Actions réalisées
- Relecture des ressources des modules déjà vus (M0-B1, M1-B1, M1-B2, M2-B1, M2-B2, M3-B1, M4-B1, M5-B1 + fiches générales) pour vérifier que la démarche correspond à ce qui a été enseigné.
- Ajout de tests de robustesse (§6.4, compétence C4) : comportement du modèle sur des entrées hors distribution et sensibilité à de petites perturbations.
- Ajout d'un `DummyClassifier` comme performance plancher, et analyse de la sensibilité du modèle retenu au seuil de décision (§5.6bis), en plus du `class_weight`.
- Traçabilité des expérimentations dans `experiments.md`, à la racine du repo.
- Avant d'attaquer §7 (interprétation client) et §8 (industrialisation), lecture des ressources correspondantes : galerie d'explicabilité (SHAP, permutation importance, détection de proxy), note de recommandation client M6-B1 (structure en 5 blocs), calibration, cheatsheet MLOps/déploiement, grille de décision stockage.

### 📝 Observations / Difficultés
- L'essentiel (split avant traitement, pipeline, gestion du déséquilibre) suivait déjà les conventions vues en cours. La relecture a fait ressortir quelques manques : tests de robustesse, baseline Dummy, seuil de décision, traçabilité des runs.
- Ces manques ont été comblés le jour même.

### ✅ Prochaines étapes
- §7.1 : explication globale et locale du modèle retenu.

## 📅 Journal de bord – Jour 8 (27/09/2026)

### 🔧 Étapes / Actions réalisées
- §7.1 : explication globale du modèle (coefficients de la régression logistique + permutation importance, les deux méthodes concordent) et explication locale sur deux clients types, avec vérification de l'additivité des contributions.
- Détection de proxy : entraînement d'un modèle auxiliaire pour retrouver le statut « senior (65 ans et plus) » à partir des seules variables du Scénario 3 (sans `age` ni `job`). ROC-AUC de 0.940 : retirer les variables sensibles ne suffit pas à empêcher leur reconstruction indirecte, et le mécanisme est identifié (le contexte économique au moment de l'appel).
- §7.2 : analyse des erreurs en fonction de la probabilité prédite et conception d'un fallback (zone de révision 0.35-0.50, audit humain mensuel par échantillonnage). Un seuil de rejet symétrique ne convenait pas, les deux types d'erreurs n'étant pas concentrés autour de 0.5.
- §7.3-7.4 : message client en 5 blocs (constat, diagnostic, recommandation, coût, décision) et synthèse de l'interprétation.

### 📝 Observations / Difficultés
- La détection de proxy est le résultat le plus parlant de l'analyse éthique : le signal résiduel supposé est maintenant mesuré et expliqué.
- L'hypothèse de départ (erreurs concentrées près du seuil) ne s'est pas vérifiée, le fallback a été adapté en conséquence.

### ✅ Prochaines étapes
- §8.1-8.2 : persistance du modèle (`.joblib` + métadonnées) et API basique (`/health`, `/predict`), sur le modèle des ressources M0-B1/M1-B2.

## 📅 Journal de bord – Jour 9 (03/10/2026)

### 🔧 Étapes / Actions réalisées
- Avant de commencer §8, relecture de mon repo M1-B2 (`app/main.py`, `schemas.py`, `middleware.py`, tests, Dockerfile, métadonnées du modèle) pour reprendre la même méthode.
- Choix de périmètre : construire directement la version complète de l'API (validation Pydantic stricte, logs structurés, tests, Docker, CI) plutôt qu'une version minimale à refaire ensuite, puisque M5 couvre déjà ces briques. Cohérent avec la feuille de route, qui conseille de ne pas « construire deux fois ».
- §8.1 : sauvegarde du pipeline complet (préprocessing + modèle) du Scénario 3 avec ses métadonnées (5 clés obligatoires + métriques CV et holdout + colonnes attendues).
- §8.2 : API FastAPI (`/health`, `/info`, `/predict`), schéma Pydantic aligné sur les colonnes du Scénario 3, réutilisation du code de préprocessing du notebook sans duplication.
- Middleware de logs structurés (Loguru) : un `request_id` par requête, aucune donnée personnelle dans les logs.
- 10 tests pytest qui passent (cas valide, champ manquant, catégorie/valeur invalide, déterminisme, contrat du modèle, modalité inconnue).
- API lancée en local et testée avec `curl`.
- Dockerfile : image de 759 Mo, conteneur `healthy`, exécution en utilisateur non-root, testé depuis l'hôte.
- CI GitHub Actions : tests puis build Docker, le build est bloqué si les tests échouent.
- §8.4-8.6 : schéma d'architecture (Mermaid) et synthèse de l'industrialisation.

### 📝 Observations / Difficultés
- Point d'attention : `has_prior_contact` est dérivée de `pdays`, ce n'est pas un champ brut. L'API doit appeler la même fonction de transformation que l'entraînement, sinon les deux peuvent diverger sans que rien ne le signale.
- Chaque brique a été testée en exécution (build Docker, requêtes HTTP).

### ✅ Prochaines étapes
- §9 (suivi en production) et §8.4 (dossier d'architecture cible) dépendaient de modules pas encore disponibles au début du projet. M7-M8 étant maintenant accessibles, reprise par §9.

## 📅 Journal de bord – Jour 10 (04/10/2026)

### 🔧 Étapes / Actions réalisées
- Relecture des ressources sur le suivi en production (M5-B1 pour Prometheus/Grafana, M6-B1 pour la dérive et la calibration) et de mon repo M5-B1 pour garder les mêmes conventions (résolution par nom de service, healthchecks, dashboard provisionné par fichier).
- Instrumentation de l'API avec `prometheus-fastapi-instrumentator` + un compteur métier `bank_marketing_predictions_total` (par classe prédite). Les 10 tests passent toujours après l'ajout.
- Inspection de `/metrics` en local avant d'écrire les requêtes PromQL.
- `docker-compose.yml` (API + Prometheus + Grafana) et dashboard Grafana propre au projet (`bank_marketing.json`, 4 panneaux : vie/`up`, latence, ratio de prédictions, volume de requêtes), provisionné en YAML/JSON.
- Stack lancée avec `docker compose up -d --build`, les 3 services `Up`/`(healthy)` dans `docker compose ps`.
- Environ 150 requêtes envoyées sur `/predict` pour générer du trafic, puis vérification via l'API HTTP de Prometheus (`/api/v1/query`) et de Grafana (`/api/search`, `/api/datasources`) que le dashboard affiche de vraies valeurs : `up`=1, p95=18ms, ratio et volume de prédictions réels.
- Test de la « panne muette » (mini-cours M6-B1) : arrêt du conteneur API (`docker compose stop api`), `up` tombe à 0 en quelques secondes alors que les compteurs applicatifs ne signalent aucune erreur. Après redémarrage, `up` repasse à 1.

### 📝 Observations / Difficultés
- Les buckets d'histogramme par défaut (0.1/0.5/1s) sont trop larges pour une latence de quelques ms : utilisation des buckets « highr » pour le panneau de latence.
- Dans le script de génération de trafic, le tableau bash était exécuté par `zsh`, où les tableaux commencent à l'indice 1. Avec `RANDOM % 3`, certaines requêtes partaient avec un corps vide et renvoyaient des 422. Le problème venait du script et non de l'API : corrigé en forçant l'exécution avec `bash`.

### ✅ Prochaines étapes
- Analyse de dérive batch et rédaction de §9.1 à §9.5.

## 📅 Journal de bord – Jour 11 (07/10/2026)

### 🔧 Étapes / Actions réalisées
- `src/drift_detection.py` (PSI/KS/Chi²) et `src/calibration.py` (ECE), avec les mêmes signatures que dans le mini-cours M6-B1.
- Analyse de dérive batch dans le notebook sur un découpage temporel des données (mars-juin vs juillet-décembre), faute de données de production (limite indiquée dans le notebook). Suivi en complément de l'AUC et de la calibration (ECE) entre les deux périodes.
- Rédaction de §9.1 (métriques à surveiller, infrastructure live), §9.2 (boucle de rétroaction), §9.3 (plan de réentraînement à plusieurs déclencheurs, jamais automatique), §9.4 (robustesse en exploitation, en lien avec les fragilités vues en §6.4), §9.5 (synthèse + note de recommandation en 5 blocs comme en §7.3).
- Notebook exécuté en entier (`nbconvert --execute`) : 0 erreur sur 97 cellules.

### 📝 Observations / Difficultés
- La cellule §9.1-9.5 du template était une seule cellule (comme §8.1-8.6 le 03/10). En insérant §9.1, les sous-sections 9.2 à 9.5 ont failli être écrasées ; récupérées avec `git show HEAD:...`.
- Séparation claire dans le notebook entre monitoring live (Prometheus/Grafana) et analyse batch (dérive/calibration, qui nécessite les vraies cibles et se fait donc en différé). Un indicateur batch n'a pas sa place dans un panneau Grafana.

### ✅ Prochaines étapes
- Annexes (glossaire, sources, décisions techniques) et dossier d'architecture cible complet (§8.4).

## 📅 Journal de bord – Jour 12 (08/10/2026)

### 🔧 Étapes / Actions réalisées
- Relecture complète du notebook : formulations simplifiées et harmonisées de §6 à §9.
- En-tête du notebook mis à jour (date de début, dernière mise à jour) et date de réception du dataset corrigée (04/08/2026) dans §0.5 et §2.
- Dates de `datasheet.md` et `experiments.md` recalées sur les dates réelles des travaux (benchmark et correctif `clone()` le 05/09, baseline Dummy le 19/09).
- `demo/index.html` : page statique de démonstration de l'API (hors livrable), avec un formulaire qui appelle `/predict`. Boutons de remplissage aléatoire (générique, orienté « oui », orienté « non », avec des combinaisons macro-économiques réelles testées sur l'API) et bouton qui tire une vraie ligne du jeu de test (`demo/test_sample.json`, 200 lignes, même split qu'en §4/§5, avec l'étiquette réelle pour comparer).
- Deux relectures complètes du code sur l'ensemble du projet : une première sur `demo/`, une seconde sur `src/`, `app/`, `tests/`, Docker/CI/infra.
- 7 corrections après la première relecture : bug du PSI (renvoyait 0.0 sans avertissement quand la référence avait une variance quasi nulle, au lieu de détecter la dérive), refus des champs inattendus côté API (`extra="forbid"`, sinon un client pouvait envoyer `age`/`job` sans être rejeté), exception interne plus renvoyée au client, validation du `X-Request-ID` envoyé par le client, Dockerfile et dépendances alignés sur les versions Python/scikit-learn de l'entraînement, dossier `demo/` documenté dans le README.
- La seconde relecture a montré que la correction du PSI était incomplète : elle couvrait le cas d'une seule valeur, mais pas les variables à faible cardinalité (ex. `has_prior_contact`, binaire), qui retombaient dans le même bug. Corrigé avec un découpage par valeur exacte (et non par quantile) pour toute variable à moins de 10 valeurs distinctes. Autres corrections : header `X-Request-ID` absent quand une exception n'était pas convertie en `HTTPException`, 2 tests de non-régression ajoutés (`extra="forbid"` et validation du `request_id`), `numpy` épinglé dans `requirements-api.txt`.
- Avec le PSI corrigé, deux chiffres du notebook ont changé (`emp.var.rate` et `nr.employed`, sous-estimés à cause du bug) : mis à jour dans §9.1bis et `NOTES-PERSO.md` (13.74 et 13.63 au lieu de 0.200 et 2.730). La conclusion (dérive forte, AUC stable) reste la même.
- Relecture des mini-cours liés à la Phase 8 (M7-B1 audit, M7-B2 « 3 architectures comparées », M8-B1 « schéma d'architecture cible » + « risques éthiques/AI Act », M8-B2 « 5 arbitrages techniques » + « fiche de décision » + « questions jury prévues ») et du dossier du cas Devalle (M8-B2, en binôme) comme modèle de format.
- `dossier_architecture.md` à la racine du repo (8 sections + annexe, même format que le dossier M8-B2) : contexte, 5 arbitrages techniques (ML classique retenu, SLM/LLM/RAG/agents et zero-shot écartés avec justification), architecture cible (schéma Mermaid avec un nœud de décision pour la zone grise 0.35-0.50), on-premise vs cloud souverain (recommandation : hébergeur français plutôt qu'un hyperscaler américain, pour la conformité et la souveraineté), acteurs à consulter (SI, DPO, métier), dimensionnement, coûts, 5 questions de jury anticipées.
- Dimensionnement mesuré avec `psutil` : pipeline de 2.5 Ko, latence d'inférence de 1.05 ms (200 runs), ~121 Mo de mémoire au chargement (essentiellement le runtime Python/pandas/sklearn, pas le modèle), image Docker de 724 Mo.
- §8.4 du notebook : ajout d'une sous-section « 8.4bis Architecture cible » (schéma condensé + synthèse, renvoi vers le dossier complet). §8.5/§8.6 n'étaient plus à jour (Python 3.11 au lieu de 3.12, monitoring §9 noté « pas encore couvert ») : corrigés.
- Annexes du notebook : glossaire (15 termes), sources et bibliographie, décisions techniques détaillées (5 justifications : F1 sur la classe positive, class_weight plutôt que SMOTE, seuil fixe pour comparer les modèles vs zone grise en exploitation, bug `clone()`, bug PSI).
- Vérification complète depuis un clone propre du dépôt distant : environnement recréé de zéro, 13 tests pytest OK, les deux notebooks exécutés en entier sans erreur, image Docker reconstruite et testée.
- Passage en revue des 96 cellules pour repérer les restes de template : le tableau §7.2 (fallback) contenait encore les exemples du canevas, alors que les décisions réelles (zone grise 0.35-0.50, HITL) étaient déjà citées ailleurs (§9.4, dossier d'architecture). Remplacé par le contenu réel.
- Vérification des chiffres repris d'un fichier à l'autre : §8.2/§8.6 indiquaient encore « 10 tests », corrigé en 13 ; §8.3 indiquait « interface non construite », précisé avec `demo/`.
- Retrait des consignes du canevas dans le notebook principal (🎯 objectifs, 💡 astuces, ⚠️ avertissements génériques, lignes « Compétences : Cx », légende des symboles) : une trentaine de cellules, traitées une par une pour ne retirer que les consignes. Le §7.2 faisait doublon avec un tableau plus détaillé quelques cellules plus bas : remplacé par une courte introduction qui y renvoie. L'introduction du §8 contenait une checklist du canevas qui ne correspondait pas au projet (`/train` et MLflow, non utilisés) : retirée.
- Notebook relancé en entier après chaque série de modifications (0 erreur), 13 tests toujours OK.

### 📝 Observations / Difficultés
- Plusieurs dates avaient été notées au moment de la rédaction et non au moment des travaux ; l'historique Git a servi de référence pour les corriger.
- Aucun des deux bugs du PSI (variance nulle, puis faible cardinalité) n'était visible à la simple lecture du code : ils sont apparus en testant sur les données.
- Le dossier M8-B2 (binôme) contient une section « Décisions de groupe » qui n'a pas de sens pour un cas individuel : retirée, les 7 autres sections sont conservées.
- Le trou du §7.2 est passé inaperçu plusieurs jours parce que le contenu existait ailleurs (notes perso, autres sections) ; il ne manquait qu'à l'endroit prévu dans le notebook.
- Certaines analyses utilisent aussi les emojis 🎯/💡/⚠️, d'où le retrait des consignes cellule par cellule plutôt qu'un remplacement automatique.

### ✅ Prochaines étapes
- Restent : fusion notebook + journal de bord, PowerPoint.

## 📅 Journal de bord – Jour 13 (09/10/2026)

### 🔧 Étapes / Actions réalisées
- Corrections de 3 erreurs dans le notebook, chaque chiffre recalculé avant de modifier le texte :
  1. §5.5 : le réglage des hyperparamètres portait sur le Scénario 1 (Random Forest), écarté en §6. Refait sur le Scénario 3 (régression logistique, le modèle retenu). Le `C` par défaut (1.0) est déjà quasi optimal (F1 0.4499 à C=1 contre 0.4504 à C=100).
  2. §9.1bis : l'AUC/ECE de dérive étaient calculés sur `df_fe` (dataset complet), qui contient des lignes vues à l'entraînement, donc une fuite. Restreint à `X_test` : AUC 0.7985 → 0.8000 (au lieu de 0.7916 → 0.7895). La conclusion « stable » ne change pas.
  3. §6.4 : le test « +5 % sur `nr.employed` » partait d'une valeur déjà au maximum du train, donc la perturbation sortait de la plage connue. Ajout d'une vérification de plage. `nr.employed` ne varie que de ~5,3 % sur toute sa plage dans le train, donc un « +5 % relatif » en sort presque toujours : noté comme point de méthode dans le notebook.
- `src/train.py` ne contenait qu'un docstring, alors que le sujet demande le réentraînement en CI. Écriture d'un script autonome : chargement des données, reconstruction du pipeline Scénario 3, évaluation sur le holdout, sauvegarde du modèle + métadonnées. L'option `--check-only` réentraîne et vérifie un F1 minimal sans rien écrire sur le disque.
- Le notebook (§8.1) écrasait `models/*.joblib` à chaque `Run All` : remplacé par une vérification de cohérence. Une nouvelle version du modèle ne peut désormais être enregistrée que via `python -m src.train`.
- Job `retrain-check` ajouté à la CI : lance `python -m src.train --check-only` à chaque push pour vérifier que le pipeline se réentraîne de zéro, sans republier le modèle. Validé sur un run GitHub Actions.
- `python src/train.py` cassait l'import `from src.preprocess import ...` (`src` absent du path) : la commande documentée partout est maintenant `python -m src.train`.
- La mesure de latence ligne par ligne (§6.4) faisait monter le fichier à 22 Mo à cause d'un avertissement sklearn/joblib répété des centaines de fois (bruit, pas une erreur). Filtré dans la cellule d'imports, le fichier est revenu à ~710 Ko.
- README mis à jour (structure, section « Réentraînement du modèle »).
- §5.8 (nouvelle section) : effet de `class_weight="balanced"` sur les probabilités. ROC-AUC presque identique (0.8014 avec, 0.8018 sans), mais `balanced` décalibre fortement les probabilités : moyenne prédite de 0.39 pour un taux réel de 0.113 (quasi parfait sans `balanced`). Cela explique la colonne `proba_moy_predite` (0.34 à 0.81) de la partie équité : c'est un effet du rééquilibrage, pas un bug.
- §6.1bis (nouvelle section) : comparaison S3/S4 à nombre d'appels égal (top 10 % / 20 %) au lieu du seuil 0.5. Captures presque identiques (44,9 % vs 45,6 % à 10 %, 64,4 % vs 64,0 % à 20 %) : le choix de S3 (§6.2) repose sur l'explicabilité et la vitesse, pas sur une meilleure capture.
- Conclusion équité par profession : le ratio `taux_selection / taux_reel` reste dans une fourchette serrée (1.6× à 2.3×) sur les 12 métiers, le ciblage suit donc les différences réelles de souscription, sans discrimination résiduelle par métier. L'écart se situe chez les 65+ : taux réel le plus élevé (50 %) et 96,4 % de sollicitation, soit un problème de calibration/saturation sur un petit groupe (n=110) plutôt qu'un biais au sens du DI. Décision : revue manuelle obligatoire pour tout segment à plus de 90 % de sollicitation et moins de 150 personnes, indépendamment du modèle. Appeler 96 % des 65+ pour un produit d'épargne relève d'un choix commercial et éthique, pas du modèle.
- Lien GitHub ajouté dans l'introduction du notebook.
- `/review` complet sur les 4 commits du jour (après les retours de la tutrice) : 4 vrais problèmes trouvés, tous corrigés.
  1. Le job CI `build` ne dépendait que de `test`, pas de `retrain-check` — une régression de F1 aurait quand même laissé publier l'image Docker malgré ce que dit le README. Fixé (`needs: [test, retrain-check]`).
  2. §9.4/§9.5 citaient encore l'ancien ECE avec fuite (0.273→0.288) alors que le fix du jour en §9.1bis l'avait corrigé à 0.269→0.290 (test set seul) — deux chiffres différents pour la même mesure. Fixé.
  3. En-tête du notebook resté au 08/10/2026 alors que tous les correctifs du jour y étaient déjà. Fixé.
  4. Code mort en §6.1bis (une variable clonée jamais utilisée). Retirée.

### 📝 Observations / Difficultés
- Corriger beaucoup de chiffres d'affilée risquait de rendre le texte autour incohérent : le notebook a été relancé en entier avec vérification `errors == 0` après chaque lot de 2-3 corrections.
- Presque chaque correction a eu un effet de bord imprévu (import cassé, fichier qui gonfle), repéré grâce à ces ré-exécutions complètes.

### ✅ Prochaines étapes
- Les 4 commits du jour passés en revue complète, pas seulement relus — ça a payé : les bugs trouvés étaient tous des oublis de propagation (un fix fait à un endroit, pas répercuté ailleurs).
- Reste en attente : réponse de la tutrice sur "B.3 (art 22) vide" (pas retrouvé vide dans la version envoyée).
- Restent : fusion notebook + journal de bord, PowerPoint.